In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('adi-dev')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/22 16:56:36 WARN Utils: Your hostname, lionix, resolves to a loopback address: 127.0.1.1; using 192.168.1.7 instead (on interface wlo1)
26/08/22 16:56:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/22 16:56:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
from datetime import date

from pyspark.sql import functions as F

from adi.pipeline import TrialBalancePipeline
from adi.config.settings import TABLE_PATHS
from adi.enrichments import (
    TransformationManager, 
    ReferenceManager
)
from adi.io import (
    CsvStore, 
    TrialBalanceRepository, 
    TransformationRepository, 
    ReferenceRepository
)

from finmap import FinMapClient


def display_df(df):
    display(df.toPandas())

In [3]:
business_dt = date(2025, 3, 31)

store = CsvStore(spark, table_paths=TABLE_PATHS)

repository = TrialBalanceRepository(store)

transformation_repository = TransformationRepository(store)
transformation_manager = TransformationManager(transformation_repository)

reference_repository = ReferenceRepository(store)
reference_manager = ReferenceManager(reference_repository)

finmap = FinMapClient.from_csv(
    spark=spark,
    metadata_path='data/reference/mapping_meta.csv',
    data_path='data/reference/mapping_data.csv',
)

pipeline = TrialBalancePipeline(
    business_dt = business_dt,
    finmap=finmap,
    repository=repository,
    reference_manager=reference_manager,
    transformation_manager=transformation_manager,
    
    
)

In [4]:
pipeline.run()

(datetime.date(2025, 3, 31), 3)

In [5]:
df_source = repository.read_source(business_dt=business_dt)

display_df(df_source)

,BATCH_ID,EXTRACT_DT,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,SRC_APP_NM,SRC_RECORD_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,...,SRC_ACCT_CATEGORY,SRC_ACCT_TYPE,NORM_ACCT_SIGN,SRC_CLIENT_ID,SRC_CLIENT_NM,SRC_MEASURE_NM,SRC_MEASURE_CCY_CD,SRC_MEASURE_TRANS_AMT,POSTING_MEASURE_CCY_CD,CPTY_REF_ID
0,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-1,NKC,4613,100011,...,BS,Assets,DEBIT,308282,,src_prev_day_bal_amt,CAD,3424081.950000000000,CAD,375545
1,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-1,NKC,4613,100011,...,BS,Assets,DEBIT,308282,,src_current_day_debit,CAD,0E-12,CAD,375545
2,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-1,NKC,4613,100011,...,BS,Assets,DEBIT,308282,,src_current_day_credit,CAD,-709.880000000000,CAD,375545
3,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-1,NKC,4613,100011,...,BS,Assets,DEBIT,308282,,src_current_day_eod_balance,CAD,3423372.070000000000,CAD,375545
4,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-1,NKC,4613,100011,...,BS,Assets,DEBIT,308282,,src_back_valued_adjustment,CAD,0E-12,CAD,375545
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-16,NSA,4032,220051,...,BS,Liabilities,CREDIT,,,src_current_day_debit,USD,0E-12,USD,
92,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-16,NSA,4032,220051,...,BS,Liabilities,CREDIT,,,src_current_day_credit,USD,0E-12,USD,
93,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-16,NSA,4032,220051,...,BS,Liabilities,CREDIT,,,src_current_day_eod_balance,USD,-27108.500000000000,USD,
94,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-16,NSA,4032,220051,...,BS,Liabilities,CREDIT,,,src_back_valued_adjustment,USD,0E-12,USD,


In [8]:
df_staging = repository.read_staging(business_dt, batch_id='3')

display_df(df_staging)

,BATCH_ID,EXTRACT_DT,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,SRC_APP_NM,DATACLASS,SRC_RECORD_ID,STAGING_ID,SRC_ENTITY_CD,...,SRC_MEASURE_CCY_CD,SRC_MEASURE_TRANS_AMT,POSTING_MEASURE_CCY_CD,POSTING_MEASURE_NM,MEASURE_TYPE,POSTING_MEASURE_FUNC_CCY_CD,POSTING_MEASURE_TRANS_AMT,FX_RATE,POSTING_MEASURE_FUNC_AMT,CR_DR_EVALUATOR
0,3,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-1,STG-250331-250331-3-1,NKC,...,CAD,3424081.950000000000,CAD,PREVIOUS_DAY_BALANCE,REPORTABLE,CAD,3424081.950000000000,1.000000000000,3424081.950000000000,DEBIT
1,3,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-1,STG-250331-250331-3-2,NKC,...,CAD,0E-12,CAD,CURRENT_DAY_DEBIT_BALANCE,REPORTABLE,CAD,0E-12,1.000000000000,0E-12,DEBIT
2,3,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-1,STG-250331-250331-3-3,NKC,...,CAD,-709.880000000000,CAD,CURRENT_DAY_CREDIT_BALANCE,REPORTABLE,CAD,-709.880000000000,1.000000000000,-709.880000000000,DEBIT
3,3,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-1,STG-250331-250331-3-4,NKC,...,CAD,3423372.070000000000,CAD,CURRENT_DAY_EOD_BALANCE,REPORTABLE,CAD,3423372.070000000000,1.000000000000,3423372.070000000000,DEBIT
4,3,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-1,STG-250331-250331-3-5,NKC,...,CAD,0E-12,CAD,BACK_VALUE_ADJUSTED_BALANCE,REPORTABLE,CAD,0E-12,1.000000000000,0E-12,DEBIT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,3,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-16,STG-250331-250331-3-92,NSA,...,USD,0E-12,USD,CURRENT_DAY_DEBIT_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,CREDIT
92,3,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-16,STG-250331-250331-3-93,NSA,...,USD,0E-12,USD,CURRENT_DAY_CREDIT_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,CREDIT
93,3,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-16,STG-250331-250331-3-94,NSA,...,USD,-27108.500000000000,USD,CURRENT_DAY_EOD_BALANCE,REPORTABLE,USD,-27108.500000000000,1.000000000000,-27108.500000000000,CREDIT
94,3,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,TRIAL_BALANCE,rec-16,STG-250331-250331-3-95,NSA,...,USD,0E-12,USD,BACK_VALUE_ADJUSTED_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,CREDIT
